# pycnlm — Overview & Quick Tour

**`pycnlm`** is a unified toolkit for **Continuous Non-Linear Manifold (CNLM)** methods
applied to combinatorial optimization — primarily **Boolean Satisfiability (SAT)** and
**Maximum Satisfiability (MaxSAT)**.

It bundles three complementary components:

| Component | Import alias | What it does |
|-----------|--------------|--------------|
| **CNLM-Langevin** | `pycnlm.langevin` | A fast–slow stochastic-differential-equation (SDE) solver for SAT / MaxSAT from DIMACS `.cnf` / `.wcnf`. |
| **HOBO Reducers** | `pycnlm.hobo` | A library of *quadratization* methods that compile high-degree pseudo-Boolean objectives down to QUBO form. |
| **AdaptCNLM** | `pycnlm.adapt` | Symmetry-based **qubit-count reduction** + embedding for quantum annealers (Chimera / Pegasus). |

This notebook is the map. Each component then has its own deep-dive tutorial:

* **`01_langevin_sat_maxsat.ipynb`** — solve SAT & MaxSAT, tune the SDE, plot convergence.
* **`02_hobo_reducers.ipynb`** — build HOBOs, apply every reducer, benchmark them.
* **`03_adaptcnlm_qubit_reduction.ipynb`** — detect symmetry, compress qubits, solve.

> All tutorials are **self-contained**: the sample instances live in `examples/data/` and
> every solver call uses small sizes so the notebooks run in seconds on a laptop CPU.

## 1. Installation

From PyPI (once published) or from a local checkout:

In [2]:
# Pick ONE of these (commented so this cell is safe to run):
# !pip install pycnlm                 # core install
# !pip install "pycnlm[benchmark]"    # + classical baselines (python-sat, etc.)
# !pip install "pycnlm[all]"          # everything (dwave, torch baselines, ...)
#
# Local editable install from the repo root:
# !pip install -e .
print("See the commented commands above; this notebook assumes pycnlm is importable.")

See the commented commands above; this notebook assumes pycnlm is importable.


In [3]:
import pycnlm
print("pycnlm version:", pycnlm.__version__)
print("version_info   :", pycnlm.__version_info__)

pycnlm version: 0.1.0
version_info   : (0, 1, 0)


## 2. The public API at a glance

Everything you usually need is re-exported at the top level, so `from pycnlm import X` works
for the common objects. The deeper machinery stays reachable under the three sub-packages.

In [4]:
import textwrap

public = [n for n in pycnlm.__all__ if not n.startswith("__")]
print("Top-level public names ({}):\n".format(len(public)))
print(textwrap.fill(", ".join(public), width=88))

Top-level public names (25):

langevin, hobo, adapt, SATInstance, parse_cnf_file, DimacsParseError, parse_dimacs_cnf,
parse_dimacs_wcnf, parse_dimacs_auto, LangevinSATInstance, MaxSATInstance,
build_literal_matrix, CNLMLangevinSolver, SolverConfig, SolveResult, solve_sat_file,
solve_maxsat_file, solve_folder, HOBO, QuadResult, QuadratizationMethod,
SymmetryDetector, OrbitBasedEncoder, CliqueBasedEncoder, ClusterBasedEncoder


In [5]:
# The three sub-package aliases
print("pycnlm.langevin ->", pycnlm.langevin.__name__)
print("pycnlm.hobo     ->", pycnlm.hobo.__name__)
print("pycnlm.adapt    ->", pycnlm.adapt.__name__)

pycnlm.langevin -> pycnlm.core.LangevinCNLM.cnlm_langevin
pycnlm.hobo     -> pycnlm.core.HOBOReducers
pycnlm.adapt    -> pycnlm.core.AdaptCNLM


## 3. Thirty-second taste of each component

Below is the *smallest possible* call into each of the three components, just to confirm
your install works and to show the shape of each API. The dedicated notebooks explain the
parameters, the maths, and the visualizations.

In [6]:
from pathlib import Path

DATA = Path("data")
print("Sample instances shipped with these tutorials:")
for f in sorted(DATA.iterdir()):
    print(f"  {f.name:24s} {f.stat().st_size:5d} bytes")

Sample instances shipped with these tutorials:
  sample_3sat.cnf            185 bytes
  sample_maxsat.wcnf         116 bytes
  sample_symmetric.cnf       251 bytes


### 3a. CNLM-Langevin — solve a `.cnf`

In [7]:
from pycnlm import SolverConfig, solve_sat_file

cfg = SolverConfig(n_steps=400, n_chains=16, seed=0)
res = solve_sat_file(DATA / "sample_3sat.cnf", cfg)
print(f"SAT? {res.is_SAT}   satisfied {res.n_satisfied}/{res.n_clauses} clauses "
      f"in {res.runtime_s*1000:.1f} ms")

SAT? True   satisfied 8/8 clauses in 0.8 ms


### 3b. HOBO Reducers — quadratize a cubic term

In [8]:
from pycnlm import HOBO
from pycnlm.hobo import PTR_Ishikawa

# Objective with a degree-3 (cubic) monomial x0*x1*x2
h = HOBO({frozenset({0, 1, 2}): 1.0, frozenset({0}): -1.0}, n_vars=3)
print("degree before:", h.degree, "| quadratic?", h.is_quadratic)

qr = PTR_Ishikawa()(h.copy())
print("degree after :", qr.qubo.degree, "| auxiliary vars added:", qr.n_aux)

degree before: 3 | quadratic? False
degree after : 2 | auxiliary vars added: 1


### 3c. AdaptCNLM — compress qubits via symmetry/structure

In [9]:
from pycnlm import parse_cnf_file, SymmetryDetector, ClusterBasedEncoder

sat = parse_cnf_file(DATA / "sample_symmetric.cnf")
orbits = SymmetryDetector(sat).find_orbits()
_, n_qubits = ClusterBasedEncoder(sat, orbits).allocate_qubits()
print(f"original variables: {sat.num_vars}  ->  cluster-encoded qubits: {n_qubits} "
      f"({(1 - n_qubits/sat.num_vars)*100:.0f}% fewer)")

Loaded CNF file: data/sample_symmetric.cnf
  Variables: 9
  Clauses: 14
Cluster-Based Encoding: 6 qubits from 3 clusters
original variables: 9  ->  cluster-encoded qubits: 6 (33% fewer)


## 4. Where to go next

You now have a working install and have touched all three components. Continue with the
deep dives:

1. **`01_langevin_sat_maxsat.ipynb`** — the SDE solver in depth (schedules, chains,
   convergence diagnostics, MaxSAT, enhancements).
2. **`02_hobo_reducers.ipynb`** — the full quadratization catalogue, spectrum verification,
   and a head-to-head benchmark.
3. **`03_adaptcnlm_qubit_reduction.ipynb`** — symmetry detection, the three encoders,
   bit-flip post-processing, and optional D-Wave embedding.

Happy solving! 🧩